# 📈 S&P 500 Market Direction Predictor 📉

A machine learning model that predicts whether the S&P 500 index will close **higher or lower tomorrow** based on historical price data and technical indicators.

**Key skills demonstrated**
- Time series analysis with financial data
- Feature engineering from raw price data
- Backtesting methodology for model validation
- Random Forest classification for binary prediction
- Handling imbalanced classes with probability thresholds

## 1. 🎯 Business Problem & Project Goals

## Personal Motivation
As someone deeply interested in financial markets, I've always been fascinated by the challenge of predicting market movements. The intersection of quantitative finance and machine learning represents a blend of interests - using data-driven approaches to understand and potentially forecast the complex dynamics of the stock market. This project was born from that curiosity and my desire to apply ML skills to a real-world financial problem.

### Problem Statement
Financial market prediction is a classic challenge in quantitative finance. While predicting exact prices is extremely difficult, predicting *direction* (up/down) can provide valuable signals for trading strategies.

### Project Objectives
- Build a binary classifier to predict if S&P 500 closes lower or higher tomorrow
- Create meaningful technical features from raw price data
- Develop a backtesting framework to validate performance
- Achieve better-than-random accuracy (>50%)

### Why This Matters
A small edge in directional prediction, scaled at a high volume can be profitable when combined with proper risk management in trading strategies. But beyond potential profitability, this project is ultimately driven by my own self-curiosity - I wanted to explore how machine learning can uncover patterns/trends in the financial markets.

## 2. 📊 Data Acquisition
I used the `yfinance` library to fetch historical S&P 500 data up to present day.

In [ ]:
import yfinance as yf
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score
import matplotlib.pyplot as plt

# Fetch S&P 500 data
sp500 = yf.Ticker("^GSPC")
sp500 = sp500.history(period="max")

# Display first few rows
print(f"Data shape: {sp500.shape}")
print(f"Date range: {sp500.index.min()} to {sp500.index.max()}")
sp500.head()

### 🔍 Initial Data Observations
- **24,664 rows** of daily data from 1927 to 2026
- Contains Open, High, Low, Close prices plus Volume
- Also includes Dividend and Stock Split features (which are zero for the S&P 500 index)

## 3. 📊 Data Visualisation

In [ ]:
# Plot closing prices over time
sp500.plot.line(y="Close", use_index=True, figsize=(12, 6))
plt.title('S&P 500 Historical Closing Prices (1927-2026)')
plt.ylabel('Price ($)')
plt.grid(True, alpha=0.3)
plt.show()

**Key Insight:** The logarithmic scale of growth shows the power of compound returns over a long period of time.

## 4. 👨🏼‍💻 Feature Engineering: Creating the Target Variable
The core of this project is to create a **target variable** that represents tomorrow's direction.

### Methodology:
1. Remove non-predictive columns (Dividends, Stock Splits)
2. Create a 'Tomorrow' column with next day's closing price
3. Define 'Target' = 1 if tomorrow's price > today's price, else 0
4. Focus on modern era (1990 onwards) for relevance

In [ ]:
# Clean up data
del sp500["Dividends"]
del sp500["Stock Splits"]

# Create target variable
sp500["Tomorrow"] = sp500["Close"].shift(-1)
sp500["Target"] = (sp500["Tomorrow"] > sp500["Close"]).astype(int)

# Focus on modern era (1990-present)
sp500 = sp500.loc["1990-01-01":].copy()
print(f"Modern era data shape: {sp500.shape}")
print(f"Target distribution:\n{sp500['Target'].value_counts(normalize=True)}")

### 📊 Target Distribution Analysis
The dataset is fairly balanced:
- **Up days:** ~53% of trading days
- **Down days:** ~47% of trading days

This balanced distribution is ideal for classification problems.

## 5. 🌳 Initial Model: Random Forest
I started with a baseline Random Forest model using only raw price features.

In [ ]:
# Initialise model
model = RandomForestClassifier(n_estimators=100, min_samples_split=100, random_state=1)

# Split data (last 100 days for testing)
train = sp500.iloc[:-100]
test = sp500.iloc[-100:]

# Define predictors
predictors = ["Close", "Volume", "Open", "High", "Low"]

# Train and predict
model.fit(train[predictors], train["Target"])
preds = model.predict(test[predictors])
preds = pd.Series(preds, index=test.index)

# Evaluate
initial_precision = precision_score(test["Target"], preds)
print(f"Initial Model Precision: {initial_precision:.3f}")

### 📝 Initial Results
The model achieved **79% precision** on this test set - but this might be bias due to the specific time period chosen.

## 6. ✨ Robust Evaluation with Backtesting
A single train/test split is not reliable for time series. I implemented a **rolling window backtesting** framework.

In [ ]:
def predict(train, test, predictors, model):
    """Make predictions and combine with actual targets"""
    model.fit(train[predictors], train["Target"])
    preds = model.predict(test[predictors])
    preds = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["Target"], preds], axis=1)
    return combined

def backtest(data, model, predictors, start=2500, step=250):
    """
    Rolling window backtesting
    - start: initial training size (2500 days ≈ 10 years)
    - step: prediction window size (250 days ≈ 1 trading year)
    """
    all_predictions = []
    
    for i in range(start, data.shape[0], step):
        train = data.iloc[0:i].copy()
        test = data.iloc[i:(i+step)].copy()
        predictions = predict(train, test, predictors, model)
        all_predictions.append(predictions)
    
    return pd.concat(all_predictions)

# Run backtest
predictions = backtest(sp500, model, predictors)

# Evaluate
backtest_precision = precision_score(predictions["Target"], predictions["Predictions"])
print(f"Backtest Precision: {backtest_precision:.3f}")
print(f"\nPrediction distribution:\n{predictions['Predictions'].value_counts()}")
print(f"\nActual target distribution:\n{predictions['Target'].value_counts(normalize=True)}")

### Backtest Results
- **Precision dropped to 53%** - much more realistic than 79%
- The model predicts "down" (~46% of predictions) less often than actual down days occur
- This suggests the model is conservative about predicting downward moves

## 7. 🤖 Advanced Feature Engineering

To improve performance, I created **technical indicators** based on rolling windows:

### New Features Created:
1. **Price Ratios:** Current close / moving average close (for 2,5,60,250,1000-day windows)
   - Indicates whether price is above/below recent averages
   
2. **Trend Strength:** Rolling sum of past targets
   - Measures recent momentum (how many up days in the period)

### Why These Features Matter
- Price ratios capture mean reversion and trend following signals
- Trend sums capture market sentiment and momentum
- Multiple time horizons capture both short and long-term patterns


In [ ]:
# Define multiple time horizons
horizons = [2, 5, 60, 250, 1000]
new_predictors = []

for horizon in horizons:
    # Rolling averages
    rolling_averages = sp500.rolling(horizon).mean()
    
    # Price ratio feature
    ratio_column = f"Close_Ratio_{horizon}"
    sp500[ratio_column] = sp500["Close"] / rolling_averages["Close"]
    
    # Trend feature (sum of recent targets)
    trend_column = f"Trend_{horizon}"
    sp500[trend_column] = sp500.shift(1).rolling(horizon).sum()["Target"]
    
    new_predictors += [ratio_column, trend_column]

# Remove rows with NaN values (caused by rolling calculations)
sp500 = sp500.dropna()
print(f"Final dataset shape with engineered features: {sp500.shape}")
sp500.head()

## 8. ✅ Enhanced Model with Probability Thresholds

I upgraded the model and added a **probability threshold** to reduce false positives.

### Key Amendments/Improvements:
- Increased estimators (200 trees) for better stability
- Lowered `min_samples_split` to 50 for more nuanced patterns
- **60% probability threshold** instead of default 50%
  - Only predict "up" when model is highly confident
  - Accept more "down" predictions to improve precision


In [ ]:
# Enhanced model
model = RandomForestClassifier(n_estimators=200, min_samples_split=50, random_state=1)

def predict_with_threshold(train, test, predictors, model, threshold=0.6):
    """Predict using probability threshold for higher confidence"""
    model.fit(train[predictors], train["Target"])
    
    # Get probability of class 1
    pred_proba = model.predict_proba(test[predictors])[:, 1]
    
    # Apply threshold
    preds = (pred_proba >= threshold).astype(int)
    
    preds = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["Target"], preds], axis=1)
    return combined

# Override the predict function in backtest
def predict(train, test, predictors, model):
    return predict_with_threshold(train, test, predictors, model)

# Run backtest with new features and threshold
enhanced_predictions = backtest(sp500, model, new_predictors)

# Evaluate
enhanced_precision = precision_score(enhanced_predictions["Target"], enhanced_predictions["Predictions"])
print(f"Enhanced Model Precision: {enhanced_precision:.3f}")
print(f"\nPrediction distribution:\n{enhanced_predictions['Predictions'].value_counts()}")

## 9. 📚 Final Results & Analysis

### Performance Improvement
| Model | Precision | Prediction Balance |
|-------|-----------|-------------------|
| Simple Model | 53.1% | 3,944 up / 2,670 down |
| Enhanced Model | **57.7%** | 874 up / 4,739 down |

### Key Insights
1. **The threshold strategy works**: By being more selective about "up" predictions, precision improved from 53% to **57.7%**
2. **Model is now conservative**: Only predicts "up" when highly confident (only 874 up predictions vs 3,944 previously)
3. **Better than random**: 57.7% precision on up days provides a meaningful edge

### Trading Implications
- A 57.7% accuracy on up-day predictions is potentially profitable
- The model's conservatism means fewer trades, but higher quality signals
- Could be combined with other indicators for a comprehensive strategy


## 10. 📝 Conclusions & Future Improvements

### What I Learned
- Time series requires careful backtesting (simple train/test splits are misleading)
- Feature engineering from raw data significantly impacts performance
- Probability thresholds can improve precision at the cost of recall

### Limitations
- Only uses price data (absence of external factors such as macroeconomic indicators, news sentiment, etc.)
- Binary classification does not capture magnitude of moves
- Transaction costs not considered

### Future Enhancements
1. **Additional data sources:**
   - VIX (fear index)
   - Interest rates
   - Economic indicators
   - News sentiment analysis

2. **Advanced modeling:**
   - Gradient boosting (XGBoost, LightGBM)
   - Ensemble methods combining multiple models

3. **Real-world application:**
   - Build a trading simulator with transaction costs
   - Add risk management rules


---
## 📊 How to Run This Project

### Requirements
`yfinance`
`pandas`
`sklearn`
`matplotlib`

### Installation
```bash
pip install yfinance pandas sklearn matplotlib